In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import torch 
from src.config import CHECKPOINT_DIR
from scripts.common.get_device import get_available_device
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT, DATASET_ROOT
from src.Skeleton_model.stgcn import STGCN
from src.XAI.STGCN_grad_cam import STGCNGradCam

In [3]:
experiment_root = CHECKPOINT_DIR / "STGCN_V1" / "grid_search_1" / "lr_2e4"
device = get_available_device()

with open(experiment_root / "config.json", "r") as f:
    hyperparameters = json.load(f)

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train", max_people=hyperparameters["max_people"])
val_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="val", max_people=hyperparameters["max_people"])
rgb_dataset = RWF2000Dataset(DATASET_ROOT, split="val", return_rgb_frames=True, num_frames=150)
radii = compute_joint_distance_to_center_of_gravity(train_dataset)
skeleton_graph = SkeletonGraph(radii, normalisation=hyperparameters["adjacency_normalisation_mode"])
model = STGCN(skeleton_graph.A, temporal_kernel_size=hyperparameters["temporal_kernel_size"], dropout=hyperparameters["dropout"], edge_importance_weighting=hyperparameters["edge_importance_weighting"]).to(device)

checkpoint = torch.load(experiment_root / "best_model.pt", map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

Using cuda:3 with 23.15 GB free


/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


STGCN(
  (data_batch_norm): BatchNorm1d(51, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (stgcn_blocks): ModuleList(
    (0): STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(3, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (temporal_conv): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0), bias=False)
        (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): Dropout(p=0, inplace=False)
      )
      (relu): ReLU(inplace=True)
    )
    (1-3): 3 x STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(64, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (temporal_conv): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running

In [10]:
stg_gradcam = STGCNGradCam(
    model=model,
    target_layer=model.stgcn_blocks[-1],
    normalisation_mode="per_video",
)
video_num = 7

input_tensor, label = val_dataset[video_num]
input_tensor = input_tensor.unsqueeze(0).to(device)

cam, logits = stg_gradcam.generate_heatmap(input_tensor)

In [11]:
print(cam.shape)

torch.Size([150, 17, 2])
